In [36]:
import nltk
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Akash\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [37]:
documents = [

"""
Python is widely used in data science and machine learning.
Pandas helps analysts work with structured tabular datasets.
NumPy provides efficient numerical computations using arrays.
It is heavily used in scientific computing applications.
TensorFlow and PyTorch are widely used deep learning frameworks.
They help researchers build neural network models efficiently.

Climate change affects ecosystems and global temperatures.
Carbon emissions are one of the major causes of global warming.
Renewable energy sources such as solar and wind power reduce dependence on fossil fuels.
Governments are investing in sustainable transportation and clean energy systems.
""",

"""
The Taj Mahal is located in Agra, India.
It was built by Emperor Shah Jahan in memory of Mumtaz Mahal.
The monument attracts millions of tourists every year.
It is made primarily of white marble.

Paris is the capital city of France.
The Eiffel Tower attracts visitors from around the world.
The Louvre Museum contains famous artworks including the Mona Lisa.
French cuisine includes croissants, baguettes, and cheese dishes.
""",

"""
Football is one of the most popular sports worldwide.
The FIFA World Cup is organized every four years.
Lionel Messi and Cristiano Ronaldo are legendary football players.
Teams often rely on counterattacks and possession strategies.

Basketball requires teamwork, passing, and shooting accuracy.
The NBA is considered the most famous basketball league globally.
Michael Jordan inspired generations of basketball players.
Three-point shooting has become crucial in modern basketball tactics.
""",

"""
Space exploration has improved significantly in recent decades.
NASA launched several missions to study Mars and nearby planets.
Satellites help scientists monitor weather systems and communications.
Astronauts undergo years of physical and technical training.

Cybersecurity protects computer systems from malware and hacking attacks.
Encryption secures sensitive information shared online.
Hackers often exploit weak passwords and vulnerable software systems.
Organizations use authentication and firewalls to improve digital security.
""",

"""
Artificial intelligence is transforming healthcare and finance industries.
Machine learning systems can analyze large datasets efficiently.
Hospitals use AI tools to assist doctors in disease diagnosis.
Financial institutions use AI for fraud detection and risk analysis.

Ocean pollution threatens marine ecosystems worldwide.
Plastic waste harms fish, turtles, and coral reefs.
Environmental organizations promote recycling and waste reduction programs.
Governments are introducing laws to reduce single-use plastics.
"""
]

queries = [
    ("Which technology helps build neural network models?", "TensorFlow"),
    ("What reduces dependence on fossil fuels?", "Renewable energy"),
    ("Who built the Taj Mahal?", "Shah Jahan"),
    ("Which museum contains the Mona Lisa?", "Louvre"),
    ("Which basketball league is globally famous?", "NBA"),
    ("What protects systems from malware attacks?", "Cybersecurity"),
    ("What secures sensitive online information?", "Encryption"),
    ("Which library supports scientific numerical computation?", "NumPy"),
    ("What helps doctors diagnose diseases?", "AI"),
    ("What harms marine ecosystems?", "Plastic waste")
]

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

Fixed-Size chunking

In [38]:
def fixed_size_chunking(text,
                        chunk_size=25,
                        overlap=5):
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = words[i:i + chunk_size]
        chunks.append(" ".join(chunk))
        
    return chunks

Sentence-Aware Chunking

In [39]:
from nltk.tokenize import sent_tokenize

def sentence_aware_chunking(text,
                            max_words=25,
                            overlap_sentences=1):
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_len = 0
    for sentence in sentences:
        sentence_len = len(sentence.split())
        if current_len + sentence_len > max_words:
            chunks.append(" ".join(current_chunk))
            overlap = current_chunk[-overlap_sentences:]
            current_chunk = overlap.copy()
            current_len = sum(
                len(s.split()) for s in current_chunk
            )
        current_chunk.append(sentence)
        current_len += sentence_len
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks


Semantic chunking

In [40]:
from sklearn.metrics.pairwise import cosine_similarity

def semantic_chunking(text,
                      similarity_threshold=0.40,
                      min_sentences=2,
                      max_words=40):

    sentences = sent_tokenize(text)

    embeddings = embed_model.encode(sentences)

    chunks = []

    current_chunk = [sentences[0]]

    current_len = len(sentences[0].split())

    for i in range(1, len(sentences)):

        similarity = cosine_similarity(
            [embeddings[i]],
            [embeddings[i - 1]]
        )[0][0]

        sentence_len = len(sentences[i].split())

        # KEEP ADDING RELATED SENTENCES
        if (
            similarity >= similarity_threshold
            or len(current_chunk) < min_sentences
        ) and current_len + sentence_len <= max_words:

            current_chunk.append(sentences[i])

            current_len += sentence_len

        else:

            chunks.append(" ".join(current_chunk))

            current_chunk = [sentences[i]]

            current_len = sentence_len

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [ ]:
def build_vector_store(chunks):
    embeddings = embed_model.encode(chunks)
    embeddings = np.array(embeddings).astype('float32')
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index

In [ ]:
def retrieve(query,
             index,
             chunks,
             top_k=3):
    query_embedding = embed_model.encode([query])
    query_embedding = np.array(query_embedding).astype('float32')
    distances, indices = index.search(query_embedding, top_k)
    retrieved_chunks = [chunks[i] for i in indices[0]]
    return retrieved_chunks


In [ ]:
def evaluate_chunking(method_name,
                      chunk_function):
    print("\n")
    print("=" * 70)
    print(f"{method_name.upper()} CHUNKING")
    print("=" * 70)
    all_chunks = []

    # CREATE CHUNKS
    for doc in documents:
        chunks = chunk_function(doc)
        all_chunks.extend(chunks)
    print(f"\nTotal Chunks Created: {len(all_chunks)}")

    # AVERAGE CHUNK SIZE
    avg_len = np.mean([
        len(chunk.split())
        for chunk in all_chunks
    ])
    print(f"Average Chunk Length: {avg_len:.2f} words")

    # BUILD VECTOR STORE
    index = build_vector_store(all_chunks)
    top1_correct = 0
    top3_correct = 0
    
    # QUERY LOOP
    for query, expected_answer in queries:
        retrieved_chunks = retrieve(
            query,
            index,
            all_chunks,
            top_k=3
        )
        top1_hit = expected_answer.lower() in \
                   retrieved_chunks[0].lower()
        top3_hit = any(
            expected_answer.lower() in chunk.lower()
            for chunk in retrieved_chunks
        )
        if top1_hit:
            top1_correct += 1
        if top3_hit:
            top3_correct += 1

        print("\n--------------------------------------------------")
        print(f"Query: {query}")

        print(f"Expected Answer: {expected_answer}")

        print(f"Top-1 Correct: {top1_hit}")

        print(f"Top-3 Correct: {top3_hit}")

        print("\nTop Retrieved Chunk:")
        print(retrieved_chunks[0][:300])

    top1_accuracy = top1_correct / len(queries)
    top3_accuracy = top3_correct / len(queries)

    print("\n")
    print("=" * 70)

    print(f"TOP-1 ACCURACY : {top1_accuracy:.2f}")

    print(f"TOP-3 ACCURACY : {top3_accuracy:.2f}")

    print("=" * 70)

    return top1_accuracy, top3_accuracy

In [ ]:
fixed_top1, fixed_top3 = evaluate_chunking(
    "Fixed Size",
    fixed_size_chunking
)

sentence_top1, sentence_top3 = evaluate_chunking(
    "Sentence Aware",
    sentence_aware_chunking
)

semantic_top1, semantic_top3 = evaluate_chunking(
    "Semantic",
    semantic_chunking
)



FIXED SIZE CHUNKING

Total Chunks Created: 21
Average Chunk Length: 21.05 words

--------------------------------------------------
Query: Which technology helps build neural network models?
Expected Answer: TensorFlow
Top-1 Correct: False
Top-3 Correct: True

Top Retrieved Chunk:
learning frameworks. They help researchers build neural network models efficiently. Climate change affects ecosystems and global temperatures. Carbon emissions are one of the major causes

--------------------------------------------------
Query: What reduces dependence on fossil fuels?
Expected Answer: Renewable energy
Top-1 Correct: True
Top-3 Correct: True

Top Retrieved Chunk:
one of the major causes of global warming. Renewable energy sources such as solar and wind power reduce dependence on fossil fuels. Governments are investing

--------------------------------------------------
Query: Who built the Taj Mahal?
Expected Answer: Shah Jahan
Top-1 Correct: True
Top-3 Correct: True

Top Retrieved Chunk:


In [ ]:
print("\n")
print("=" * 70)
print("FINAL COMPARISON")
print("=" * 70)

print(f"Fixed-size     -> Top1: {fixed_top1:.2f} | Top3: {fixed_top3:.2f}")
print(f"Sentence-aware -> Top1: {sentence_top1:.2f} | Top3: {sentence_top3:.2f}")
print(f"Semantic       -> Top1: {semantic_top1:.2f} | Top3: {semantic_top3:.2f}")

print("=" * 70)



FINAL COMPARISON
Fixed-size     -> Top1: 0.70 | Top3: 1.00
Sentence-aware -> Top1: 0.90 | Top3: 1.00
Semantic       -> Top1: 1.00 | Top3: 1.00
